# test_single_sample.ipynb
**Name:** Numan Hussan &nbsp;&nbsp; **Roll No.:** MSDS25067

**Assignment:** DL Spring 2025 - Assignment 5 (Bonus) - Diffusion Models

This notebook loads the **trained model checkpoint** (`Saved_Models/diffusion_model.pt`)
and generates a **single image sample** purely from random Gaussian noise, by running
the reverse diffusion process. This is the notebook the instructor/TA will open during
evaluation to see your results, as required by the assignment.

In [ ]:
# If running in Google Colab, make sure model.py and the Saved_Models folder
# are in the same directory as this notebook (see Readme.txt / Colab guide).
import torch
import matplotlib.pyplot as plt

from model import Diffusion, UNet

In [ ]:
# ---- Settings ----
CHECKPOINT_PATH = "Saved_Models/diffusion_model.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

In [ ]:
# ---- Load checkpoint ----
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

img_size = checkpoint["img_size"]
timesteps = checkpoint["timesteps"]
base_channels = checkpoint["base_channels"]

model = UNet(in_channels=3, base_ch=base_channels).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

diffusion = Diffusion(timesteps=timesteps, img_size=img_size, device=DEVICE)
print("Model loaded. img_size =", img_size, " timesteps =", timesteps)

In [ ]:
# ---- Generate a single sample from pure noise ----
with torch.no_grad():
    sample = diffusion.sample(model, n_samples=1)

img = sample[0].permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.axis("off")
plt.title("Generated sample (noise -> image)")
plt.show()

In [ ]:
# ---- Alternative: explicitly create noise yourself, then pass it in ----
# (This matches the assignment wording exactly: "write a test function
# which will accept noise and create an image from it".)

noise = torch.randn(1, 3, img_size, img_size)   # create noise explicitly

with torch.no_grad():
    sample2 = diffusion.sample(model, n_samples=1, starting_noise=noise)

img2 = sample2[0].permute(1, 2, 0).cpu().numpy()
plt.figure(figsize=(4, 4))
plt.imshow(img2)
plt.axis("off")
plt.title("Generated from explicitly-created noise")
plt.show()

### Notes for the report
- Re-run the cell above a few times to generate several different samples (each
  starts from a different random noise tensor).
- For the report, also include screenshots of:
  - `outputs/forward_process.png` (forward noising visualization),
  - `outputs/loss_curve.png` (training loss curve),
  - `outputs/final_generated_samples.png` (a batch of generated images).